# U-Net Semantic Segmentation (Pixel-wise Segmentation) – PASCAL VOC Example

This notebook is an extension of a Faster R-CNN transfer learning example. It demonstrates how to use a U-Net model for semantic segmentation.

U-Net works by:
- Encoding the input image into a lower-dimensional feature space using a CNN (encoder)
- Decoding it back to the original resolution using another CNN (decoder)
- Performing pixel-level classification of objects

Compared to DeepLab v3:
- U-Net performs well with smaller datasets (commonly used in medical imaging)
- DeepLab v3 is better suited for large-scale urban datasets

This example uses the PASCAL VOC 2012 dataset.

We use the `segmentation-models-pytorch` library for pretrained U-Net.


In [ ]:
# Install dependency (run once)
!pip install segmentation-models-pytorch

## 1. Download Dataset and Visualize Samples

The PASCAL VOC dataset (~2GB) will be downloaded automatically.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import VOCSegmentation
import segmentation_models_pytorch as smp

data_root = './data'
download = True

train_dataset = VOCSegmentation(root=data_root, year='2012', image_set='train', download=download)
val_dataset = VOCSegmentation(root=data_root, year='2012', image_set='val', download=download)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
def show_image_and_mask(dataset, idx):
    img, mask = dataset[idx]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img)
    axes[0].set_title("Original image")
    axes[0].axis("off")
    axes[1].imshow(mask)
    axes[1].set_title("Ground truth mask")
    axes[1].axis("off")
    plt.show()

show_image_and_mask(train_dataset, random.randint(0, len(train_dataset)-1))
show_image_and_mask(train_dataset, random.randint(0, len(train_dataset)-1))

## 2. Data Preprocessing

- Resize images to 256×256
- Normalize images to [0,1]
- Convert masks to class indices (0–20)

Note: PASCAL VOC uses label 255 as a "void" label.

In [ ]:
image_size = 256

transform_img = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

transform_mask = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

def transform_train(image, mask):
    image = transform_img(image)
    mask = transform_mask(mask)
    mask = (mask * 255).long().squeeze(0)
    return image, mask

class VOCDatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, voc_dataset, transform=None):
        self.voc_dataset = voc_dataset
        self.transform = transform

    def __len__(self):
        return len(self.voc_dataset)

    def __getitem__(self, idx):
        image, mask = self.voc_dataset[idx]
        if self.transform:
            image, mask = self.transform(image, mask)
        return image, mask

train_dataset_transformed = VOCDatasetWrapper(train_dataset, transform=transform_train)
val_dataset_transformed = VOCDatasetWrapper(val_dataset, transform=transform_train)

## 3. Model Setup (U-Net with ResNet34 Encoder)

- Pretrained on ImageNet
- 21 output classes (20 objects + background)
- Ignore label 255 in loss

In [ ]:
batch_size = 4
train_loader = DataLoader(train_dataset_transformed, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset_transformed, batch_size=batch_size, shuffle=False)

n_classes = 21

model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=n_classes,
    activation=None
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f"Using device: {device}")

## 4. Training

In [ ]:
num_epochs = 30
best_val_loss = float('inf')

for epoch in range(1, num_epochs+1):
    model.train()
    train_loss = 0.0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item() * images.size(0)

    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_unet_voc.pth')

    scheduler.step()

print("Training finished")

## 5. Inference and Visualization

In [ ]:
model.load_state_dict(torch.load('best_unet_voc.pth', map_location=device))
model.eval()

def predict_and_plot(model, dataset, idx):
    image, true_mask = dataset[idx]

    with torch.no_grad():
        output = model(image.unsqueeze(0).to(device))
        pred_mask = torch.argmax(output, dim=1).squeeze(0)

    img_np = image.cpu().permute(1,2,0).numpy()
    true_np = true_mask.cpu().numpy()
    pred_np = pred_mask.cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Original")
    axes[1].imshow(true_np)
    axes[1].set_title("Ground Truth")
    axes[2].imshow(pred_np)
    axes[2].set_title("Prediction")
    plt.show()

predict_and_plot(model, val_dataset_transformed, random.randint(0, len(val_dataset_transformed)-1))
predict_and_plot(model, val_dataset_transformed, random.randint(0, len(val_dataset_transformed)-1))